# Ścieżka E · Preferencje wielokryterialne (hotel, swing + ROC)

**Projektowanie i badanie informatycznych systemów wspomagania decyzji** · notatnik projektu z aplikacją · szkielet do uzupełnienia

Ten notatnik prowadzi cały projekt ścieżki E w jednym miejscu, zgodnie z *Szczegółowym przewodnikiem badania preferencji i budowy aplikacji* (ścieżka E):

| Etap | Kto robi | Pliki |
|---|---|---|
| 1–4 · eksport, scoring, profil | zespół (luki `# TODO`) | `qualtrics_raw.csv` → `responses_clean.csv` → `profil.csv` |
| 5–6 · model, wybór 12 par | zespół (luki `# TODO`) | `pool.csv` |
| 7–8 · zamrożenie, kontrola kontraktu | gotowe | `cases.csv`, `session_config.json` |
| 9 · sesja z uczestnikiem | **gotowa aplikacja Streamlit** (nie edytujesz) | zapisuje `validation.csv`, `session_S01.json` |
| 10–11 · analiza walidacji | zespół | wynik do raportu |

**Aplikacja jest gotowa.** Nie piszecie interfejsu. Aplikacja czyta pliki tworzone przez ten notatnik i działa tylko wtedy, gdy spełniają one kontrakt (sekcja 8). Predykcji nie liczy sama: bierze je z zamrożonego `cases.csv`.

**Dwa tryby pracy** (komórka konfiguracji poniżej):
- `TRYB = 'przyklad'`: wymyślony uczestnik P01 z przewodnika. Wszystkie testy sprawdzają liczby kontrolne z przewodnika. Zacznij tutaj.
- `TRYB = 'wlasne'`: eksport Waszego uczestnika. Testy funkcji działają dalej, a testy przypisane do liczb P01 są pomijane: notatnik wypisuje wtedy Wasz wynik zamiast go sprawdzać.

**Ścieżka E ma więcej ogniw obliczeń niż pozostałe ścieżki**, dlatego dostajecie więcej gotowego kodu. Każda luka jest mała, ma własny test, a przy trudniejszych krokach są wskazówki.

**Pomoc AI:** [Konsultant DSS (Gem w Gemini)](https://gemini.google.com/gem/1B6czvZbCeBBuqqjSMid08BPCGsWN1qDe?usp=sharing) wyjaśnia pojęcia i błędy, pomaga zaplanować testy, ale nie uzupełnia luk `# TODO`. Każde użycie AI zapiszcie w dzienniku: cel pomocy, decyzja zespołu, sposób kontroli. Plan pracy i przykłady P01: [strona Pracowni](https://p4welrog.github.io/decyzje/).

> **Jak pracować.** Uruchamiaj komórki po kolei. W miejscach `# TODO` uzupełnij kod i usuń `raise NotImplementedError`. Test pod każdym krokiem pokazuje ✓ albo zatrzymuje się na ✗ z opisem. Nie zmieniaj liczb w testach.

## 0 · Konfiguracja i folder projektu na Dysku Google
W Colabie pliki znikają po rozłączeniu, dlatego folder roboczy leży na Dysku Google. Przy pierwszym uruchomieniu Colab poprosi o zgodę na dostęp do Dysku. Folder można udostępnić zespołowi i prowadzącym. Dla każdego trybu i każdego uczestnika powstaje osobny folder, więc przykład nie miesza się z danymi.

W plikach są tylko kod uczestnika i jego odpowiedzi. Informacja przed zgodą powinna jednak mówić, gdzie przechowujecie dane i kto ma do nich dostęp.

In [ ]:
TRYB = 'przyklad'          # 'przyklad' albo 'wlasne'
PARTICIPANT = 'P01'        # kod uczestnika (taki sam w Qualtricsie i w sesji)
SESSION_ID = 'S01'         # identyfikator sesji walidacyjnej
INSTRUCTOR_PIN = '2468'    # kod widoku prowadzącego w aplikacji; nie pokazuj go uczestnikowi
UZYJ_DYSKU = True          # w Colabie: folder projektu na Dysku Google

import os, sys, hashlib, inspect, json, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if TRYB not in ('przyklad', 'wlasne'):
    raise ValueError("TRYB musi mieć wartość 'przyklad' albo 'wlasne'")
if TRYB == 'przyklad':
    PARTICIPANT = 'P01'
if IN_COLAB and UZYJ_DYSKU:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/projekt_E'
else:
    ROOT = os.path.abspath('projekt_E')
WORKDIR = os.path.join(ROOT, 'przyklad_P01' if TRYB == 'przyklad' else PARTICIPANT)
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

def check(label, cond):
    """Test ogólny: działa w obu trybach."""
    if not bool(cond):
        raise AssertionError('✗ ' + label)
    print('✓', label)

def check_example(label, cond, value=None):
    """Test liczb przykładu P01: w trybie 'wlasne' tylko wypisuje Wasz wynik."""
    if TRYB == 'przyklad':
        check(label, cond)
    else:
        print('ℹ pominięto test przykładu:', label, '' if value is None else f'| Wasz wynik: {value}')

def close(a, b, tol=1e-6):
    return a is not None and b is not None and abs(a - b) <= tol

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def file_sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

print(f'Tryb: {TRYB} | uczestnik: {PARTICIPANT} | folder: {WORKDIR}')

## 1 · Stałe wersji podstawowej (przewodnik, rozdz. 3)
Kryteria, zakresy i kierunki ustalają prowadzący. Są **identyczne w profilowaniu i walidacji**, bo wagi swingów mają sens tylko względem tych zakresów.

In [ ]:
PROTOCOL = 'E_v1'
MODEL_VERSION = 'E_ROC_v1'
CRIT = ['cena', 'odl', 'ocena', 'pow', 'anul']
CRITERIA = {
    'cena':  dict(label='Cena za noc',          unit='zł',  worst=480, best=180, direction='min'),
    'odl':   dict(label='Odległość od centrum', unit='km',  worst=6.0, best=0.5, direction='min'),
    'ocena': dict(label='Ocena gości',          unit='/10', worst=6.5, best=9.5, direction='max'),
    'pow':   dict(label='Powierzchnia pokoju',  unit='m²',  worst=14,  best=32,  direction='max'),
    'anul':  dict(label='Okno anulowania po rezerwacji', unit='h',   worst=0,   best=48,  direction='max'),
}
CONS_PAIRS = [('cena', 'ocena'), ('odl', 'pow'), ('anul', 'odl'), ('pow', 'cena')]   # E_CONS_01…04
TIE_EPS = 0.005
# Wagi P01 z przewodnika: stałe do testów funkcji, niezależne od Waszego uczestnika
W_P01 = {'ocena': 137 / 300, 'cena': 77 / 300, 'odl': 47 / 300, 'anul': 27 / 300, 'pow': 12 / 300}
pd.DataFrame(CRITERIA).T

## 2 · Eksport z Qualtricsa (rozdz. 5 i 8)
- `przyklad`: komórka tworzy plik w układzie prawdziwego eksportu, czyli trzy wiersze nagłówkowe, wiersz Preview, wypełnienie bez zgody i właściwe P01.
- `wlasne`: wgraj eksport CSV (wartości liczbowe). Plik trafi do folderu projektu jako `qualtrics_raw.csv`. Oryginału nie zmieniamy.

Nazwy pól muszą być zgodne ze słownikiem z rozdz. 8 przewodnika: `E_CONSENT`, `E_IMP_…`, `E_RANK_…`, `E_TIE`, `E_BUDGET`, `E_CONS_01–04` (1 pierwsza, 2 druga, 3 równe), `E_LIN_…`.

In [ ]:
RAW_PATH = 'qualtrics_raw.csv'
if TRYB == 'przyklad':
    P01 = dict(E_IMP_cena=7, E_IMP_odl=4, E_IMP_ocena=5, E_IMP_pow=3, E_IMP_anul=4,
               E_RANK_cena=2, E_RANK_odl=3, E_RANK_ocena=1, E_RANK_pow=5, E_RANK_anul=4,
               E_TIE=0, E_BUDGET=400, E_CONS_01=2, E_CONS_02=1, E_CONS_03=2, E_CONS_04=2,
               E_LIN_cena=1, E_LIN_cena_x=1, E_LIN_odl=1, E_LIN_odl_x=1)
    base = dict(ResponseId='', participant_id='P01', project_path='E', protocol_version=PROTOCOL, phase='profile',
                Status=0, Finished=1, Progress=100, **{'Duration (in seconds)': 0}, E_CONSENT=1)
    rows = [{**base, 'ResponseId': 'R_prev01', 'Status': 1, **{**P01, 'E_RANK_cena': 1, 'E_RANK_ocena': 2}},
            {**base, 'ResponseId': 'R_nozgoda', 'participant_id': 'P99', 'E_CONSENT': 0, **{k: '' for k in P01}},
            {**base, 'ResponseId': 'R_P01', 'Duration (in seconds)': 742, **P01}]
    cols = list(rows[0])
    head = pd.DataFrame([[c for c in cols], [json.dumps({'ImportId': f'QID_{c}'}) for c in cols]]
                        + [[r.get(c, '') for c in cols] for r in rows], columns=cols)
    head.to_csv(RAW_PATH, index=False)
elif not os.path.exists(RAW_PATH):
    if not IN_COLAB:
        raise FileNotFoundError(f'Umieść eksport jako {os.path.join(WORKDIR, RAW_PATH)}')
    from google.colab import files
    up = files.upload()
    open(RAW_PATH, 'wb').write(next(iter(up.values())))
print('Plik źródłowy:', os.path.join(WORKDIR, RAW_PATH))

## 3 · Wczytanie i wybór wypełnienia (gotowe)
Przeczytaj warunki filtra, bo w raporcie trzeba je opisać.

In [ ]:
def load_raw(path):
    """Eksport Qualtricsa: trzy wiersze nagłówkowe, dane od czwartego."""
    return pd.read_csv(path, skiprows=[1, 2])

def select_response(raw, participant_id, protocol_version, phase='profile'):
    """Dokładnie jedno wypełnienie: Status 0 (nie Preview 1, nie Test 2), Finished 1, E_CONSENT 1, kod, wersja, faza."""
    ok = raw[(raw['Status'] == 0) & (raw['Finished'] == 1) & (raw['E_CONSENT'] == 1)
             & (raw['participant_id'] == participant_id)
             & (raw['protocol_version'] == protocol_version) & (raw['phase'] == phase)]
    if len(ok) != 1:
        raise ValueError(f'Oczekiwano 1 wypełnienia dla {participant_id}, jest {len(ok)}')
    return ok.iloc[0]

raw = load_raw(RAW_PATH)
resp = select_response(raw, PARTICIPANT, PROTOCOL)
check_example('wybrano wypełnienie R_P01 (Preview i brak zgody odrzucone)', resp['ResponseId'] == 'R_P01')
FIELDS = ([f'E_IMP_{c}' for c in CRIT] + [f'E_RANK_{c}' for c in CRIT] + ['E_TIE', 'E_BUDGET']
          + [f'E_CONS_0{i}' for i in range(1, 5)] + ['E_LIN_cena', 'E_LIN_cena_x', 'E_LIN_odl', 'E_LIN_odl_x'])
missing = [f for f in FIELDS if f not in raw.columns]
check('eksport ma wszystkie pola codebooka', not missing or print('Brakuje:', missing))
resp[['participant_id', 'protocol_version'] + FIELDS].to_frame().T.to_csv('responses_clean.csv', index=False)
resp[FIELDS].to_frame().T

## 4 · Scoring: od rangi do wag (rozdz. 6)
### 4.1 Wagi ROC
$w_i = \frac{1}{n}\sum_{k=i}^{n} \frac{1}{k}$. Dla n = 5 daje to 137/300, 77/300, 47/300, 27/300 i 12/300.

> **Wskazówka.** Lista składana po `i` od 1 do n, a w środku `sum(1 / k for k in range(i, n + 1))`.

In [ ]:
def roc_weights(n=5):
    """Lista wag ROC dla miejsc 1..n."""
    # TODO: jedna linia (lista składana)
    raise NotImplementedError

def rank_sum_weights(n=5):
    """Wagi sumy rang (analiza wrażliwości). Gotowe."""
    total = n * (n + 1) / 2
    return [(n + 1 - i) / total for i in range(1, n + 1)]

w = roc_weights(5)
check('suma wag ROC = 1', close(sum(w), 1))
check('wagi ROC = 137, 77, 47, 27, 12 (/300)', all(close(a, b / 300) for a, b in zip(w, [137, 77, 47, 27, 12])))

### 4.2 Ranking → wagi kryteriów
Waga kryterium to waga miejsca, które zajął **jego swing**, a nie waga z deklarowanej ważności. Zgłoszony remis sąsiednich miejsc (np. `E_TIE = 23`) daje obu kryteriom średnią wag tych miejsc.

> **Wskazówka.** `divmod(23, 10)` daje `(2, 3)`. Znajdź dwa kryteria o tych rangach i obu przypisz średnią ich wag.

In [ ]:
def read_ranks(row):
    """{kryterium: ranga}; brak → None. Gotowe."""
    return {c: None if pd.isna(row.get(f'E_RANK_{c}')) or row.get(f'E_RANK_{c}') == '' else int(row.get(f'E_RANK_{c}')) for c in CRIT}

def ranking_complete(ranks):
    """Każda pozycja 1–5 dokładnie raz. Gotowe."""
    return all(r is not None for r in ranks.values()) and sorted(ranks.values()) == list(range(1, len(ranks) + 1))

def assign_weights(ranks, tie=0, base=None):
    base = base or roc_weights(len(ranks))
    w = {c: base[r - 1] for c, r in ranks.items()}
    if tie:
        # TODO: rozbij tie na dwa miejsca, znajdź dwa kryteria i nadaj im średnią wagę
        raise NotImplementedError
    return w

R_P01 = {'ocena': 1, 'cena': 2, 'odl': 3, 'anul': 4, 'pow': 5}
check('ranking P01 → wagi z przewodnika', all(close(assign_weights(R_P01)[c], W_P01[c]) for c in CRIT))
check('remis miejsc 2 i 3 → 62/300 dla obu', close(assign_weights(R_P01, 23)['cena'], 62 / 300) and close(assign_weights(R_P01, 23)['odl'], 62 / 300))
check('brak albo podwójna pozycja → ranking niekompletny', not ranking_complete({**R_P01, 'pow': None}) and not ranking_complete({**R_P01, 'pow': 4}))

ranks = read_ranks(resp)
TIE = 0 if pd.isna(resp['E_TIE']) else int(resp['E_TIE'])
print('Ranking uczestnika:', ranks, '| zgłoszony remis:', TIE or 'brak')

### 4.3 Kontrola spójności i status profilu
Kody `E_CONS_0k`: 1 = pierwsza wymieniona poprawa z `CONS_PAIRS`, 2 = druga, 3 = obie równie ważne. Odpowiedź jest zgodna z rankingiem, gdy wybrany swing ma niższą (lepszą) rangę. Jeśli kontrola porównuje parę zgłoszoną jako remis, zgodna jest tylko odpowiedź „równe”. Poza remisem odpowiedź „równe” jest niezgodna.

| Sytuacja (rozdz. 10) | Status i działanie |
|---|---|
| pełny ranking, kontrole 3/4 albo 4/4 | `valid` |
| jedna zgłoszona para równych miejsc | flaga `tie_reported`, średnie wagi |
| zgodność kontroli ≤ 2/4 | `inconsistent`: wyjaśnienie i ewentualna jedna powtórka |
| brak rangi, budżetu lub wymaganej kontroli | `incomplete`: brak personalizacji |
| różnica połówek co najmniej dwukrotna (w dowolną stronę) | flaga `nonlinear_cena` / `nonlinear_odl`, model bez zmian |

In [ ]:
def consistency(ranks, codes, tie=0):
    """(liczba zgodnych, lista True/False). Kody: 1 pierwsza, 2 druga, 3 obie równie ważne."""
    tied = set()
    if tie:
        r1, r2 = divmod(int(tie), 10)
        tied = {c for c, r in ranks.items() if r in (r1, r2)}
    ok = []
    for (first, second), code in zip(CONS_PAIRS, codes):
        code = int(code)
        # TODO: para zgłoszona jako remis → zgodna tylko odpowiedź 3; poza remisem odpowiedź 3 jest niezgodna;
        #       w pozostałych przypadkach wybrany swing musi mieć niższą rangę niż niewybrany
        raise NotImplementedError
    return sum(ok), ok

def is_missing(v):
    return v is None or v == '' or (isinstance(v, float) and math.isnan(v))

def linearity_flags(row):
    """Różnica połówek co najmniej dwukrotna, niezależnie od kierunku. Gotowe."""
    return [f'nonlinear_{c}' for c in ('cena', 'odl') if row.get(f'E_LIN_{c}') in (1, 2) and row.get(f'E_LIN_{c}_x') == 2]

def profile_status(ranks, n_consistent, budget_ok=True, controls_ok=True):
    if not ranking_complete(ranks) or not budget_ok or not controls_ok:
        return 'incomplete'
    # TODO: 'valid' albo 'inconsistent' według tabeli
    raise NotImplementedError

check('P01: spójność 4/4', consistency(R_P01, [2, 1, 2, 2])[0] == 4)
check('odwrócona pierwsza kontrola → 3/4', consistency(R_P01, [1, 1, 2, 2])[0] == 3)
check('„równe” poza remisem → niezgodne', consistency(R_P01, [3, 1, 2, 2])[0] == 3)
check('remis miejsc 1–2 (ocena, cena): w kontroli cena–ocena zgodne tylko „równe”',
      consistency(R_P01, [3, 1, 2, 2], tie=12)[0] == 4 and consistency(R_P01, [2, 1, 2, 2], tie=12)[0] == 3)
check('statusy: valid / inconsistent / incomplete (brak budżetu)',
      (profile_status(R_P01, 4), profile_status(R_P01, 2), profile_status(R_P01, 4, budget_ok=False)) == ('valid', 'inconsistent', 'incomplete'))

complete = ranking_complete(ranks)
codes = [resp[f'E_CONS_0{i}'] for i in range(1, 5)]
controls_ok = not any(is_missing(c) for c in codes)
budget_ok = not is_missing(resp['E_BUDGET'])
n_cons = consistency(ranks, codes, TIE)[0] if complete and controls_ok else None
flags = linearity_flags(resp) + (['tie_reported'] if TIE else [])
status = profile_status(ranks, n_cons or 0, budget_ok, controls_ok)
check_example('status P01 = valid, spójność 4/4, bez flag', (status, n_cons, flags) == ('valid', 4, []), (status, n_cons, flags))
if status != 'valid':
    print('⚠ Profil nie jest valid: aplikacja wstrzyma sesję. Postępuj według rozdz. 10 przewodnika.')

### 4.4 `profil.csv`
Jeden wiersz na kryterium i wspólne pola profilu. Zakresy są w pliku, więc aplikacja nie ma ich wpisanych w kodzie drugi raz. Deklarowana ważność trafia do raportu, ale **nie** do reguły.

In [ ]:
BUDGET = float(resp['E_BUDGET']) if not pd.isna(resp['E_BUDGET']) else math.inf
w_roc = assign_weights(ranks, TIE) if complete else None
w_sens = assign_weights(ranks, TIE, base=rank_sum_weights()) if complete else None
w_base = {c: 1 / len(CRIT) for c in CRIT}
profil = pd.DataFrame([dict(participant_id=PARTICIPANT, criterion=c, label=CRITERIA[c]['label'], unit=CRITERIA[c]['unit'],
                            worst=CRITERIA[c]['worst'], best=CRITERIA[c]['best'], direction=CRITERIA[c]['direction'],
                            rank=ranks[c], weight_roc=round(w_roc[c], 9) if w_roc else None,
                            weight_roc_300=f'{round(w_roc[c] * 300, 6):g}/300' if w_roc else '',
                            weight_rank_sum=round(w_sens[c], 9) if w_sens else None,
                            importance_1_7=resp[f'E_IMP_{c}'], budget_pln=BUDGET, cancel_definition='hours_after_booking', tie=TIE,
                            consistency=f'{n_cons}/4' if complete else '', flags=';'.join(flags),
                            status=status, model_version=MODEL_VERSION) for c in CRIT])
profil = profil.sort_values('rank', na_position='last')
profil.to_csv('profil.csv', index=False)
check_example('wagi ROC P01 zgodne z przewodnikiem', w_roc is not None and all(close(w_roc[c], W_P01[c]) for c in CRIT))
profil[['criterion', 'rank', 'importance_1_7', 'weight_roc', 'weight_roc_300', 'status']]

## 5 · Model (rozdz. 7 i 10)
### 5.1 Funkcja wartości
$v = \frac{x - \text{najgorszy}}{\text{najlepszy} - \text{najgorszy}}$, przycięte do [0, 1], z flagą `out_of_range`. Wzór obsługuje oba kierunki.

In [ ]:
def value(x, criterion):
    """Zwraca (v w [0, 1], out_of_range)."""
    c = CRITERIA[criterion]
    # TODO: v, przycięcie do [0, 1], krotka (v_przycięte, czy_poza_zakresem)
    raise NotImplementedError

check('cena 250 zł → v = 0,767', close(value(250, 'cena')[0], 0.7667, 1e-4))
check('odległość 5,4 km → v = 0,109', close(value(5.4, 'odl')[0], 0.1091, 1e-4))
check('ocena 9,8 → (1, out_of_range)', value(9.8, 'ocena') == (1.0, True))
check('cena 500 zł → (0, out_of_range)', value(500, 'cena') == (0.0, True))

### 5.2 Suma ważona i predykcja dla pary
Najpierw działa **filtr budżetu** (rozdz. 11): oferta droższa niż budżet odpada przed liczeniem U, a oferta równa budżetowi jest dopuszczalna. Gdy dopuszczalna jest tylko jedna oferta, wynik to ta oferta z flagą `budget_only`. Gdy żadna, wynik to `NO_PREDICTION`. Wynik to `A`, `B`, `TIE` albo `NO_PREDICTION`. Profil bez wag (`weights is None`) daje `NO_PREDICTION`. Wartości spoza zakresu są przycinane i zwracane jako `out_of_range`.

> **Wskazówka.** Kolejność: obie oferty poza budżetem → dokładnie jedna poza budżetem (wygrywa druga, `budget_only=True`) → U obu ofert, lista `out_of_range` i porównanie U z progiem `eps`.

In [ ]:
def utility(offer, weights):
    """(U, {kryterium: w·v}, [kryteria poza zakresem]). Gotowe."""
    contrib, out = {}, []
    for c in CRIT:
        v, oor = value(offer[c], c)
        contrib[c] = weights[c] * v
        if oor:
            out.append(c)
    return sum(contrib.values()), contrib, out

def predict_pair(a, b, weights, budget=math.inf, eps=TIE_EPS):
    """dict(pred, U_A, U_B, budget_only, out_of_range)."""
    res = dict(pred=None, U_A=None, U_B=None, budget_only=False, out_of_range=[])
    if weights is None:
        res['pred'] = 'NO_PREDICTION'
        return res
    a_ok, b_ok = a['cena'] <= budget, b['cena'] <= budget
    # TODO: filtr budżetu (dwa przypadki), potem U obu ofert, out_of_range i porównanie z progiem eps
    raise NotImplementedError

V01_A = dict(cena=250, odl=5.4, ocena=8.4, pow=16, anul=30)
V01_B = dict(cena=350, odl=1.6, ocena=7.7, pow=29, anul=12)
r = predict_pair(V01_A, V01_B, W_P01, 400)
check('V01, wagi P01: U = 0,564 i 0,475 → A', close(r['U_A'], 0.564, 5e-4) and close(r['U_B'], 0.475, 5e-4) and r['pred'] == 'A')
check('V01, równe wagi: 0,449 i 0,543 → B', predict_pair(V01_A, V01_B, w_base, 400)['pred'] == 'B')
check('wkład oceny gości w A = 0,2892', close(utility(V01_A, W_P01)[1]['ocena'], 0.2892, 1e-4))
check('oferta 430 zł przy budżecie 400 → B z flagą budget_only', predict_pair(dict(V01_A, cena=430), V01_B, W_P01, 400)['pred'] == 'B'
      and predict_pair(dict(V01_A, cena=430), V01_B, W_P01, 400)['budget_only'])
check('oferta równa budżetowi jest dopuszczalna', not predict_pair(dict(V01_A, cena=400), V01_B, W_P01, 400)['budget_only'])
check('ocena 9,8 → out_of_range', predict_pair(dict(V01_A, ocena=9.8), V01_B, W_P01, 400)['out_of_range'] == ['ocena'])
check('obie poza budżetem → NO_PREDICTION', predict_pair(dict(V01_A, cena=450), dict(V01_B, cena=460), W_P01, 400)['pred'] == 'NO_PREDICTION')
check('identyczne oferty → TIE', predict_pair(V01_A, V01_A, W_P01)['pred'] == 'TIE')
check('profil bez wag → NO_PREDICTION', predict_pair(V01_A, V01_B, None)['pred'] == 'NO_PREDICTION')

### 5.3 Wyjaśnienie (gotowe)
Aplikacja pokazuje ten tekst z wkładem cech **dopiero po zebraniu wszystkich wyborów**. Wykres w notatniku służy analizie błędów.

In [ ]:
def explain(a, b, weights):
    top = sorted(CRIT, key=lambda c: -weights[c])[:2]
    ua, ca, _ = utility(a, weights); ub, cb, _ = utility(b, weights)
    win = 'A' if ua > ub else 'B'
    wins = [CRITERIA[c]['label'].lower() for c in top if (ca[c] > cb[c]) == (win == 'A')]
    loses = [CRITERIA[c]['label'].lower() for c in CRIT if c not in top and (ca[c] > cb[c]) != (win == 'A')]
    t = f"Według Twojego rankingu najwięcej waży {CRITERIA[top[0]]['label'].lower()}, a potem {CRITERIA[top[1]]['label'].lower()}. "
    t += f"Hotel {win} ma wyższe dopasowanie (U = {max(ua, ub):.3f} wobec {min(ua, ub):.3f})"
    t += f" i wygrywa w: {', '.join(wins)}." if wins else '.'
    if loses:
        t += f" Drugi hotel jest lepszy w: {', '.join(loses)}, ale te poprawy ustawiono niżej w rankingu."
    return t

def plot_contrib(a, b, weights, title='Dopasowanie do profilu tej osoby (nie jakość hotelu)'):
    _, ca, _ = utility(a, weights); _, cb, _ = utility(b, weights)
    fig, ax = plt.subplots(figsize=(6, 2.2)); left = np.zeros(2)
    for c in sorted(CRIT, key=lambda c: -weights[c]):
        vals = np.array([ca[c], cb[c]]); ax.barh(['A', 'B'], vals, left=left, label=CRITERIA[c]['label']); left += vals
    ax.set_title(title); ax.invert_yaxis(); ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout(); plt.show()

print(explain(V01_A, V01_B, W_P01))
plot_contrib(V01_A, V01_B, W_P01)

## 6 · Pula 24 par i stała reguła wyboru 12 (rozdz. 11)
Kandydatami są V01–V22. V23 i V24 służą wyłącznie do kontroli filtra. Regułę stosujemy po obliczeniu profilu, a przed sesją (rozdz. 15):
1. Dla V01–V22 oblicz wynik ROC i równych wag z budżetem uczestnika.
2. Wyłącz pary z `TIE` lub `NO_PREDICTION` w którymkolwiek modelu, z danymi poza zakresami oraz rozstrzygane wyłącznie przez budżet (`budget_only`).
3. Weź najwyżej 6 par, w których modele się różnią, w kolejności numerów.
4. Uzupełnij do 12 parami zgodnymi w kolejności numerów. Gdy zgodnych jest za mało, nie dobieraj dalszych rozróżniających.

In [ ]:
POOL_RAW = [[[250,5.4,8.4,16,30],[350,1.6,7.7,29,12]],[[200,2.2,8.3,28,12],[320,3.7,7.6,24,42]],[[300,1.3,7.2,27,30],[400,3.5,9.2,22,18]],
 [[400,2.1,9.2,27,30],[180,3.8,8.0,22,18]],[[250,1.5,8.8,30,6],[350,3.2,7.5,22,42]],[[380,4.0,7.9,31,42],[310,2.7,8.5,15,6]],
 [[280,5.5,9.3,31,36],[360,1.1,8.0,23,24]],[[400,0.9,7.4,20,12],[300,5.6,8.1,27,36]],[[230,2.1,9.1,14,30],[320,3.3,8.0,29,18]],
 [[400,2.1,8.8,32,6],[240,3.7,7.7,24,42]],[[240,3.2,8.0,16,6],[360,4.5,7.3,32,42]],[[400,1.2,7.1,24,30],[210,3.1,9.4,20,18]],
 [[360,0.8,7.3,19,18],[280,4.7,8.2,26,36]],[[400,5.5,8.6,28,36],[200,0.6,7.6,21,24]],[[210,2.2,9.0,15,48],[320,3.9,7.8,29,24]],
 [[210,5.3,9.0,18,30],[350,0.9,7.7,30,12]],[[360,0.8,9.3,21,24],[300,4.8,7.1,27,42]],[[390,3.8,8.0,23,42],[270,2.5,8.7,29,0]],
 [[400,2.5,9.0,27,6],[190,4.2,7.9,21,42]],[[210,5.8,8.8,25,0],[340,0.7,8.1,20,42]],[[210,1.5,7.6,20,12],[400,4.8,8.3,26,24]],
 [[390,4.1,9.1,30,18],[290,3.0,7.0,14,42]],[[430,1.0,8.6,26,24],[330,3.0,7.6,20,24]],[[300,2.0,7.8,22,12],[460,0.8,9.0,28,48]]]
pool = pd.DataFrame([{'pair_id': f'V{i + 1:02d}', **{f'A_{c}': a[k] for k, c in enumerate(CRIT)}, **{f'B_{c}': b[k] for k, c in enumerate(CRIT)}}
                     for i, (a, b) in enumerate(POOL_RAW)])
pool.to_csv('pool.csv', index=False)

def offers(row):
    return {c: row[f'A_{c}'] for c in CRIT}, {c: row[f'B_{c}'] for c in CRIT}

CANDIDATES = [f'V{i:02d}' for i in range(1, 23)]

def classify_pool(pool, w_personal, w_baseline, budget):
    out = []
    for _, row in pool.iterrows():
        a, b = offers(row)
        p, q = predict_pair(a, b, w_personal, budget), predict_pair(a, b, w_baseline, budget)
        cls = None  # TODO: V23/V24 → 'test filtra'; krok 2 reguły (TIE, NO_PREDICTION, budget_only, out_of_range); potem: czy predykcje się różnią
        out.append(dict(pair_id=row['pair_id'], pred_personal=p['pred'], pred_baseline=q['pred'], U_A=p['U_A'], U_B=p['U_B'],
                        budget_only=p['budget_only'], out_of_range=';'.join(p['out_of_range']), cls=cls))
    return pd.DataFrame(out)

def select_cases(classified, n_total=12, n_diff_max=6):
    # TODO: najwyżej 6 różnicujących w kolejności numerów, potem wspólne do 12; posortowana lista
    raise NotImplementedError

_cl = classify_pool(pool, W_P01, w_base, 400)
check('wagi P01: różnicujące V01, V03, V06, V10, V11, V16', list(_cl.loc[_cl.cls == 'różnicująca', 'pair_id']) == ['V01', 'V03', 'V06', 'V10', 'V11', 'V16'])
check('wagi P01: V20 = TIE; V23 i V24 tylko do testu filtra (budget_only przy 400 zł)', _cl.set_index('pair_id').loc['V20', 'pred_personal'] == 'TIE'
      and set(_cl.loc[_cl.cls == 'test filtra', 'pair_id']) == {'V23', 'V24'} and _cl.set_index('pair_id').loc[['V23', 'V24'], 'budget_only'].all())
check('przy budżecie 480 zł V24 nadal nie trafia do walidacji', 'V24' not in select_cases(classify_pool(pool, W_P01, w_base, 480)))
check('budżet 350 zł: tylko 6 par spełnia regułę (rozdz. 15)', len(select_cases(classify_pool(pool, W_P01, w_base, 350))) == 6)
check('wagi P01: wybór V01–V11 oraz V16', select_cases(_cl) == [f'V{i:02d}' for i in range(1, 12)] + ['V16'])

cl = classify_pool(pool, w_roc, w_base, BUDGET)
selected = select_cases(cl)
n_diff = int(cl.loc[cl.pair_id.isin(selected), 'cls'].eq('różnicująca').sum())
print(f'Uczestnik {PARTICIPANT}: wybrane pary {", ".join(selected)} | różnicujące: {n_diff}')
if len(selected) < 12:
    print(f'⚠ Tylko {len(selected)} par spełnia regułę przy budżecie {BUDGET:g} zł (pula jest przygotowana dla budżetu ok. 400 zł). '
          'Nie podnoś budżetu i nie dobieraj par samodzielnie: pokaż to prowadzącym przed sesją (rozdz. 15).')
if n_diff < 6:
    print('⚠ Mniej niż 6 par różnicujących: opisz to w raporcie jako własność testu.')

### 6.1 Analiza wrażliwości (przed sesją)
Wagi sumy rang to nie trzeci model. Sprawdzają, czy predykcje zależą od arbitralnej formuły wag.

In [ ]:
sens = {pid: predict_pair(*offers(pool[pool.pair_id == pid].iloc[0]), w_sens, BUDGET)['pred'] for pid in selected}
changed = {pid: sens[pid] for pid in selected if sens[pid] != cl.set_index('pair_id').loc[pid, 'pred_personal']}
check_example('suma rang zmienia predykcję w V03 (TIE) i V10 (B)', changed == {'V03': 'TIE', 'V10': 'B'}, changed)
print('Zmiany przy wagach sumy rang:', changed or 'brak')

## 7 · Zamrożenie przed sesją
Opisy ofert, predykcje i wyjaśnienia trafiają do `cases.csv`. Aplikacja niczego nie liczy sama.

Zamrożenie zapisuje `cases.csv` i `session_config.json` ze skrótem SHA-256. Skrót wpisz do dziennika **przed sesją**. W trybie `wlasne` notatnik nie nadpisze zamrożonych przypadków: jeśli po zamrożeniu zmienisz dane albo kod, komórka zatrzyma się z błędem.

In [ ]:
# Opcjonalnie (rozszerzenie, np. własny interfejs): zapis przetestowanych funkcji do model.py.
# Aplikacja kursu nie potrzebuje tego pliku.
EKSPORT_MODELU = False
if EKSPORT_MODELU:
    CONSTS = ['CRIT', 'CRITERIA', 'CONS_PAIRS', 'TIE_EPS', 'MODEL_VERSION']
    FUNCS = [roc_weights, rank_sum_weights, read_ranks, ranking_complete, assign_weights, consistency,
             linearity_flags, profile_status, value, utility, predict_pair, explain]
    src = ['"""model.py – wygenerowane z notatnika ścieżki E; nie edytuj ręcznie."""', 'import math', 'import pandas as pd', '']
    src += [f'{name} = {globals()[name]!r}' for name in CONSTS] + ['']
    src += [inspect.getsource(fn) for fn in FUNCS]
    open('model.py', 'w', encoding='utf-8').write('\n'.join(src))
    import importlib
    sys.path.insert(0, WORKDIR)
    if 'model' in sys.modules: del sys.modules['model']
    import model
    check('model.py daje ten sam wynik dla V01', close(model.predict_pair(V01_A, V01_B, W_P01, 400)['U_A'], 0.564, 5e-4))

In [ ]:
if not status.startswith('valid'):
    raise RuntimeError(f'Status profilu {status}: nie zamrażamy przypadków i nie prowadzimy sesji (rozdz. 6).')
rng = np.random.default_rng(20261020)                       # ziarno zapisz w dzienniku
cases = pool[pool.pair_id.isin(selected)].merge(cl[['pair_id', 'pred_personal', 'pred_baseline', 'U_A', 'U_B']], on='pair_id')
cases['pred_rank_sum'] = cases.pair_id.map(sens)
cases['display_order'] = rng.permutation(len(cases)) + 1
cases['A_position'] = rng.choice(['left', 'right'], len(cases))
cases['model_version'] = MODEL_VERSION
cases['params'] = 'w=' + ';'.join(f'{c}:{w_roc[c]:.6f}' for c in CRIT) + f'|budget={BUDGET:g}|eps={TIE_EPS}'

def offer_md(row, side):
    o = offers(row)[0 if side == 'A' else 1]
    fmt = lambda v: f'{v:g}'.replace('.', ',')
    return f'#### Hotel {side}\n\n' + '\n'.join(f'- **{CRITERIA[c]["label"]}:** {fmt(o[c])} {CRITERIA[c]["unit"]}' for c in CRIT)

def explain_full(row):
    a, b = offers(row)
    _, ca, _ = utility(a, w_roc); _, cb, _ = utility(b, w_roc)
    parts = '; '.join(f'{CRITERIA[c]["label"].lower()} {ca[c]:.3f} / {cb[c]:.3f}' for c in sorted(CRIT, key=lambda c: -w_roc[c]))
    return explain(a, b, w_roc) + f' Wkład cech do dopasowania (A / B): {parts}. To dopasowanie do Twojego profilu, a nie jakość hotelu.'

cases.insert(0, 'case_id', cases.pair_id)
cases['left_text'] = [offer_md(r, 'A' if r.A_position == 'left' else 'B') for _, r in cases.iterrows()]
cases['right_text'] = [offer_md(r, 'B' if r.A_position == 'left' else 'A') for _, r in cases.iterrows()]
cases['left_code'] = np.where(cases.A_position == 'left', 'A', 'B')
cases['right_code'] = np.where(cases.A_position == 'left', 'B', 'A')
cases['explanation'] = [explain_full(r) for _, r in cases.iterrows()]
new_csv = cases.to_csv(index=False).encode()
new_hash = hashlib.sha256(new_csv).hexdigest()

if TRYB == 'wlasne' and os.path.exists('session_config.json'):
    frozen = json.load(open('session_config.json', encoding='utf-8'))
    if frozen['frozen_sha256'] != new_hash:
        raise RuntimeError('Przypadki są już zamrożone, a nowe obliczenia dają inny cases.csv. Nie nadpisuję. '
                           'Jeśli zmiana jest zamierzona i nastąpiła PRZED sesją, usuń świadomie session_config.json '
                           'i opisz to w dzienniku; po sesji wymaga to nowej wersji modelu i nowego zestawu.')
    print('Zamrożenie bez zmian:', new_hash)
else:
    open('cases.csv', 'wb').write(new_csv)
    json.dump(dict(contract='E_v1', participant_id=PARTICIPANT, session_id=SESSION_ID, frozen_sha256=new_hash,
                   frozen_at=now_iso(), model_version=MODEL_VERSION, seed=20261020, instructor_pin=INSTRUCTOR_PIN,
                   profile_ok=True, profile_note=status, title='Wybór hotelu na jedną noc',
                   left_button='Wybieram hotel po lewej', right_button='Wybieram hotel po prawej',
                   code_labels={'A': 'hotel A', 'B': 'hotel B'},
                   intro='Wyobraź sobie, że rezerwujesz hotel na jedną noc podczas prywatnego wyjazdu do dużego miasta. Hotele różnią się '
                         'tylko pięcioma cechami. Pozostałe warunki są takie same: ten sam termin, pokój dla jednej osoby, łazienka w pokoju, '
                         'śniadanie niewliczone. Okno anulowania to liczba godzin od rezerwacji, w których można ją bezpłatnie anulować. '
                         'Nie ma poprawnych odpowiedzi.'),
              open('session_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('Zamrożono cases.csv | SHA-256:', new_hash)
FROZEN_HASH = file_sha256('cases.csv')
cases.sort_values('display_order')[['display_order', 'pair_id', 'A_position', 'pred_personal', 'pred_baseline', 'pred_rank_sum']]

## 8 · Kontrola kontraktu z aplikacją (gotowe)
Aplikacja wymaga określonych plików i kolumn. Komórka sprawdza to samo co aplikacja przy starcie. Jeśli tutaj wszystko jest ✓, aplikacja się uruchomi.

In [ ]:
CONTRACT = {
    'profil.csv': ['participant_id', 'criterion', 'label', 'unit', 'worst', 'best', 'direction', 'rank', 'weight_roc',
                   'budget_pln', 'status', 'model_version'],
    'cases.csv': ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code', 'pred_personal',
                  'pred_baseline', 'model_version', 'params', 'explanation', 'A_position'],
}
for name, cols in CONTRACT.items():
    missing = [c for c in cols if c not in pd.read_csv(name).columns]
    check(f'{name}: wymagane kolumny', not missing or print('Brakuje:', missing))
cfg = json.load(open('session_config.json', encoding='utf-8'))
check('session_config.json: kontrakt E_v1 i zgodny skrót cases.csv', cfg['contract'] == 'E_v1' and cfg['frozen_sha256'] == file_sha256('cases.csv'))
check('każda para ma wyjaśnienie i oba opisy', pd.read_csv('cases.csv')[['left_text', 'right_text', 'explanation']].notna().all().all())
check('pred_personal i pred_baseline tylko z dozwolonych wartości', pd.read_csv('cases.csv')[['pred_personal', 'pred_baseline']].isin(['A', 'B', 'TIE', 'NO_PREDICTION']).all().all())

## 9 · Aplikacja i sesja z uczestnikiem
Komórka z kodem aplikacji jest zwinięta. **Nie edytuj jej.** Zapisuje plik `app.py` w folderze projektu.

**Przebieg sesji** (rozdz. 11):
1. Uczestnik widzi kolejne pary hoteli z pełnymi opisami i wybiera bez podpowiedzi. Każdy wybór trafia od razu do `validation.csv`.
2. Po wszystkich wyborach aplikacja pokazuje rekomendacje i wyjaśnienia z wkładem cech, a potem zbiera jedną ocenę dopasowania i jedną ocenę użyteczności całej serii, obie 1–7 (`session_S01.json`).
3. Widok prowadzącego (panel boczny, kod `INSTRUCTOR_PIN`) pokazuje kontrakt, zamrożone predykcje i log. Nie otwieraj go przy uczestniku.

Odświeżenie strony nie gubi danych: aplikacja wznawia sesję od pierwszej pary bez zapisanego wyboru.

In [ ]:
#@title Kod aplikacji – gotowy, nie edytuj { display-mode: "form" }
APP_SOURCE = r'''"""Sesja walidacyjna – ścieżki A, B, C i E (wybór jednej z dwóch opcji).

Gotowa aplikacja kursu: zespół jej NIE edytuje. Aplikacja niczego nie przewiduje sama:
opisy opcji, predykcje i wyjaśnienia pochodzą z zamrożonego cases.csv utworzonego w notatniku.

Uruchomienie:  streamlit run app.py -- --data <folder z plikami projektu>
Folder musi zawierać: cases.csv, profil.csv i session_config.json (tworzy je notatnik).
Aplikacja zapisuje: validation.csv (po każdym wyborze) i session_<id>.json (oceny serii).
"""
import argparse
import hashlib
import json
import os
import sys
from datetime import datetime, timezone

import pandas as pd
import streamlit as st

APP_VERSION = 'pred_app_v1'
REQUIRED_CASES = ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code',
                  'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation']
REQUIRED_CFG = ['contract', 'participant_id', 'session_id', 'frozen_sha256', 'instructor_pin', 'profile_ok',
                'title', 'intro', 'left_button', 'right_button', 'code_labels']


def data_dir():
    p = argparse.ArgumentParser()
    p.add_argument('--data', default='.')
    return os.path.abspath(p.parse_known_args(sys.argv[1:])[0].data)


DATA = data_dir()
path = lambda name: os.path.join(DATA, name)
now_iso = lambda: datetime.now(timezone.utc).isoformat(timespec='seconds')
sha256 = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()


def load_project():
    """Wczytuje pliki i sprawdza kontrakt. Zwraca (cfg, cases, błędy, ostrzeżenia)."""
    errors, warnings = [], []
    for name in ['cases.csv', 'profil.csv', 'session_config.json']:
        if not os.path.exists(path(name)):
            errors.append(f'Brak pliku {name} w folderze {DATA}. Uruchom notatnik do sekcji „Zamrożenie”.')
    if errors:
        return {}, None, errors, warnings
    cfg = json.load(open(path('session_config.json'), encoding='utf-8'))
    missing = [k for k in REQUIRED_CFG if k not in cfg]
    if missing:
        errors.append(f'session_config.json: brak pól {", ".join(missing)}')
    cases = pd.read_csv(path('cases.csv'), dtype={'case_id': str})
    missing = [c for c in REQUIRED_CASES if c not in cases.columns]
    if missing:
        errors.append(f'cases.csv: brak kolumn {", ".join(missing)}')
    if sha256(path('cases.csv')) != cfg.get('frozen_sha256'):
        errors.append('cases.csv różni się od wersji zamrożonej (SHA-256 nie zgadza się z session_config.json).')
    if not cfg.get('profile_ok', False):
        errors.append(f'Profil nie pozwala na personalizację: {cfg.get("profile_note", "")}')
    if not errors:
        bad = set(cases.pred_personal.astype(str)) - set(cfg['code_labels']) - {'TIE', 'NO_PREDICTION'}
        if bad:
            errors.append(f'cases.csv: nieznane kody predykcji {bad}')
        cases = cases.sort_values('display_order').reset_index(drop=True)
    return cfg, cases, errors, warnings


def read_log():
    if os.path.exists(path('validation.csv')) and os.path.getsize(path('validation.csv')) > 0:
        return pd.read_csv(path('validation.csv'), dtype={'session_id': str, 'case_id': str})
    return pd.DataFrame(columns=['session_id', 'case_id'])


def append_log(row):
    log = pd.concat([read_log(), pd.DataFrame([row])], ignore_index=True)
    tmp = path('validation.csv.tmp')
    log.to_csv(tmp, index=False)
    os.replace(tmp, path('validation.csv'))       # zapis atomowy


st.set_page_config(page_title='Sesja walidacyjna', page_icon='🧭', layout='wide')
cfg, cases, errors, warnings = load_project()
SESSION = str(cfg.get('session_id', 'S01'))
SESSION_JSON = path(f'session_{SESSION}.json')

with st.sidebar:
    st.caption(f'{APP_VERSION} · kontrakt {cfg.get("contract", "?")}')
    pin = st.text_input('Kod prowadzącego', type='password')
    instructor = bool(cfg) and pin != '' and pin == str(cfg.get('instructor_pin'))

if instructor:
    st.title('Widok prowadzącego')
    for e in errors:
        st.error(e)
    for w in warnings:
        st.warning(w)
    if not errors:
        st.success(f'Kontrakt spełniony · cases.csv zgodny z zamrożeniem · SHA-256 {cfg["frozen_sha256"][:16]}…')
        st.subheader('Profil')
        st.dataframe(pd.read_csv(path('profil.csv')), hide_index=True)
        st.subheader('Zamrożone przypadki')
        st.dataframe(cases[['display_order', 'case_id', 'left_text', 'right_text', 'pred_personal', 'pred_baseline']], hide_index=True)
    log = read_log()
    n = int((log.session_id.astype(str) == SESSION).sum()) if len(log) else 0
    st.subheader(f'Log sesji {SESSION}: {n} zapisanych wyborów')
    st.dataframe(log, hide_index=True)
    if os.path.exists(path('validation.csv')):
        st.download_button('Pobierz validation.csv', open(path('validation.csv'), 'rb').read(), 'validation.csv', 'text/csv')
    if os.path.exists(SESSION_JSON):
        st.json(json.load(open(SESSION_JSON, encoding='utf-8')))
    st.stop()

st.title(cfg.get('title', 'Sesja walidacyjna'))
if errors:
    st.error('Aplikacja nie jest gotowa do sesji. Poproś prowadzącego o sprawdzenie (widok prowadzącego w panelu bocznym).')
    st.stop()

log = read_log()
done = log[log.session_id.astype(str) == SESSION]
remaining = cases[~cases.case_id.isin(set(done.case_id.astype(str)))]
if os.path.exists(SESSION_JSON):
    st.success('Sesja zakończona. Dziękujemy za udział!')
    st.caption('Dla prowadzącego: próbną sesję można powtórzyć komórką „↺ Próbna sesja od początku” w notatniku.')
    st.stop()

if 'started' not in st.session_state:
    st.session_state.started = len(done) > 0
if not st.session_state.started:
    st.write(cfg['intro'])
    st.write(f'Zobaczysz {len(cases)} sytuacji. W każdej wybierz jedną z dwóch możliwości. '
             'Wyjaśnienia działania systemu zobaczysz dopiero po zapisaniu wszystkich wyborów.')
    if st.button('Rozpocznij', type='primary'):
        st.session_state.started = True
        st.session_state.started_at = now_iso()
        st.rerun()
    st.stop()

if len(remaining):
    row = remaining.iloc[0]
    k = len(done) + 1
    st.progress((k - 1) / len(cases), text=f'Sytuacja {k} z {len(cases)}')
    st.caption(cfg['intro'])
    cl, cr = st.columns(2)
    for col, txt in ((cl, row.left_text), (cr, row.right_text)):
        with col:
            with st.container(border=True):
                st.markdown(txt)

    def choose(position):
        code = row.left_code if position == 'left' else row.right_code
        rec = dict(participant_id=cfg['participant_id'], session_id=SESSION, case_id=str(row.case_id),
                   display_order=int(row.display_order), chosen_position=position, answer=code, answered_at=now_iso(),
                   model_version=row.model_version, params=row.params, app_version=APP_VERSION)
        rec.update({c: row[c] for c in cases.columns if c.startswith('pred_')})
        append_log(rec)

    b1, b2 = st.columns(2)
    b1.button(cfg['left_button'], key=f'L_{row.case_id}', on_click=choose, args=('left',), use_container_width=True)
    b2.button(cfg['right_button'], key=f'R_{row.case_id}', on_click=choose, args=('right',), use_container_width=True)
    st.stop()

st.progress(1.0, text='Wszystkie wybory zapisane')
st.header('Jak system opisuje Twoje wybory')
st.write('Teraz, po zapisaniu wszystkich decyzji, pokazujemy rekomendacje systemu i ich uzasadnienie. '
         'Rekomendacja opisuje działanie modelu, a nie poprawność Twojej decyzji.')
ans = dict(zip(done.case_id.astype(str), done.answer))
labels = cfg['code_labels']
for _, row in cases.iterrows():
    mine = ans.get(str(row.case_id))
    with st.expander(f'Sytuacja {int(row.display_order)}: wybrano „{labels.get(mine, mine)}”'):
        c1, c2 = st.columns(2)
        c1.markdown(row.left_text); c2.markdown(row.right_text)
        rec = row.pred_personal
        if rec in labels:
            st.write(f'**System rekomendowałby: {labels[rec]}.** {row.explanation}')
        else:
            st.write(f'System nie rozstrzyga tej sytuacji ({rec}). {row.explanation}')

st.header('Ocena całej serii')
fit = st.slider('Na ile rekomendacje były dopasowane do Twoich wyborów? (1 = wcale, 7 = bardzo)', 1, 7, 4)
useful = st.slider('Na ile wyjaśnienia były dla Ciebie użyteczne? (1 = wcale, 7 = bardzo)', 1, 7, 4)
if st.button('Zakończ sesję', type='primary'):
    json.dump(dict(participant_id=cfg['participant_id'], session_id=SESSION, rating_fit=int(fit), rating_usefulness=int(useful),
                   started_at=st.session_state.get('started_at', ''), finished_at=now_iso(), n_choices=len(done),
                   cases_sha256=cfg['frozen_sha256'], app_version=APP_VERSION),
              open(SESSION_JSON, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    st.rerun()
'''
open('app.py', 'w', encoding='utf-8').write(APP_SOURCE)
print('Zapisano app.py w', WORKDIR)

In [ ]:
#@title ▶ Uruchom aplikację w Colabie { display-mode: "form" }
URUCHOM = False  #@param {type:"boolean"}
import subprocess, socket, time
def _port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0
if URUCHOM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)
    if not _port_open(8501):
        subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', os.path.join(WORKDIR, 'app.py'),
                          '--server.port', '8501', '--server.headless', 'true',
                          '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
                          '--', '--data', WORKDIR],
                         stdout=open(os.path.join(WORKDIR, 'streamlit.log'), 'w'), stderr=subprocess.STDOUT)
        for _ in range(60):
            if _port_open(8501): break
            time.sleep(1)
    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(8501, anchor_text='▶ Otwórz aplikację w nowej karcie')
    else:
        print('Aplikacja: http://localhost:8501')
else:
    print('Zaznacz URUCHOM i uruchom komórkę ponownie. Jeśli link nie działa, przeprowadź sesję lokalnie (komórka niżej).')

In [ ]:
#@title ↺ Próbna sesja od początku { display-mode: "form" }
PROBA_OD_POCZATKU = False      #@param {type:"boolean"}
TO_BYLA_TYLKO_PROBA = False    #@param {type:"boolean"}
import shutil
_sid = json.load(open('session_config.json', encoding='utf-8'))['session_id']
_files = [f for f in ['validation.csv', f'session_{_sid}.json'] if os.path.exists(f)]
if not PROBA_OD_POCZATKU:
    print('Po zakończonej sesji aplikacja pokazuje tylko podziękowanie. Aby powtórzyć próbną sesję, zaznacz PROBA_OD_POCZATKU i uruchom komórkę.')
elif not _files:
    print('Brak logu sesji: aplikacja zacznie od początku.')
elif TRYB == 'wlasne' and not TO_BYLA_TYLKO_PROBA:
    print("⚠ Tryb 'wlasne': log może zawierać prawdziwą sesję uczestnika. Jeśli to była tylko próba przed sesją, "
          'zaznacz też TO_BYLA_TYLKO_PROBA. Po sesji z uczestnikiem logu nie przenosimy.')
else:
    _dst = os.path.join('proby', datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(_dst, exist_ok=True)
    for f in _files:
        shutil.move(f, os.path.join(_dst, f))
    print('Przeniesiono', ', '.join(_files), 'do', os.path.join(WORKDIR, _dst), '(nic nie usunięto). Odśwież kartę aplikacji.')
    if TRYB == 'wlasne':
        print('Zapisz w dzienniku: data, kto prowadził próbę i że log próby przeniesiono do', _dst)

**Sesja lokalna (zalecana przy uczestniku).** Colab może się rozłączyć w trakcie sesji. Pewniej jest uruchomić aplikację na laptopie:
1. Uruchom komórkę poniżej i pobierz `pakiet_sesji_E.zip`. Paczka zawiera `app.py`, `profil.csv`, `cases.csv` i `session_config.json`.
2. Na laptopie: `pip install streamlit pandas`, potem w rozpakowanym folderze `streamlit run app.py -- --data .`
3. Po sesji skopiuj `validation.csv` i `session_S01.json` do folderu projektu na Dysku i przejdź do sekcji 10 w trybie `wlasne`.

In [ ]:
import zipfile
with zipfile.ZipFile('pakiet_sesji_E.zip', 'w') as z:
    for f in ['app.py', 'profil.csv', 'cases.csv', 'session_config.json']:
        z.write(f)
print('Zapisano', os.path.join(WORKDIR, 'pakiet_sesji_E.zip'))
if IN_COLAB:
    from google.colab import files
    files.download('pakiet_sesji_E.zip')

## 10 · Analiza walidacji (rozdz. 11)
- `przyklad`: notatnik tworzy log w formacie aplikacji z **syntetycznymi** odpowiedziami. Plik `validation.csv` zapisany przez aplikację w folderze przykładu nie jest tu używany.
- `wlasne`: notatnik czyta `validation.csv` i `session_S01.json` zapisane przez aplikację.

Funkcja `summarize` jest gotowa i taka sama jak w ścieżkach B–D. Przeczytaj ją, bo raport musi opisać pokrycie, wspólny zbiór i podział 2×2.

In [ ]:
UNAMBIGUOUS = lambda s: s.notna() & ~s.isin(['TIE', 'NO_PREDICTION'])

def summarize(df, pred='pred_personal', base='pred_baseline', ans='answer'):
    """Podsumowanie walidacji wg przewodnika.

    - pokrycie: liczba prób z jednoznaczną predykcją (bez TIE i NO_PREDICTION),
    - trafność obu modeli na WSPÓLNYM zbiorze prób z odpowiedzią i jednoznaczną predykcją obu,
    - podział 2×2: oba trafiły / tylko indywidualny / tylko bazowy / żaden,
    - liczba prób wspólnego zbioru, w których modele się różnią.
    """
    common = df[df[ans].notna() & UNAMBIGUOUS(df[pred]) & UNAMBIGUOUS(df[base])]
    hp = common[pred] == common[ans]
    hb = common[base] == common[ans]
    return dict(
        n_trials=len(df), n_answered=int(df[ans].notna().sum()),
        coverage_personal=int(UNAMBIGUOUS(df[pred]).sum()),
        coverage_baseline=int(UNAMBIGUOUS(df[base]).sum()),
        n_common=len(common), hits_personal=int(hp.sum()), hits_baseline=int(hb.sum()),
        both=int((hp & hb).sum()), only_personal=int((hp & ~hb).sum()),
        only_baseline=int((~hp & hb).sum()), none=int((~hp & ~hb).sum()),
        n_differ=int((common[pred] != common[base]).sum()))

def report(s, name_p='model indywidualny', name_b='model bazowy'):
    pct = lambda k, m: f'{k}/{m} = {100*k/m:.0f}%' if m else f'{k}/{m}'
    print(f"Pokrycie: {name_p} {s['coverage_personal']}/{s['n_trials']}, {name_b} {s['coverage_baseline']}/{s['n_trials']}")
    print(f"Trafność na wspólnym zbiorze: {name_p} {pct(s['hits_personal'], s['n_common'])}, "
          f"{name_b} {pct(s['hits_baseline'], s['n_common'])}")
    print(f"2×2: oba {s['both']}, tylko indywidualny {s['only_personal']}, tylko bazowy {s['only_baseline']}, żaden {s['none']}")
    print(f"Próby wspólnego zbioru, w których modele się różnią: {s['n_differ']}/{s['n_common']}")
    print('TIE i NO_PREDICTION nie są ani trafieniem, ani pomyłką. Wynik opisuje test jednej osoby.')

In [ ]:
_t = pd.DataFrame({'pred_personal': ['A', 'B', 'TIE', 'A', 'A'],
                   'pred_baseline': ['A', 'A', 'B', 'NO_PREDICTION', 'B'],
                   'answer':        ['A', 'B', 'B', 'A', None]})
_s = summarize(_t)
check('pokrycie liczy tylko jednoznaczne predykcje', (_s['coverage_personal'], _s['coverage_baseline']) == (4, 4))
check('wspólny zbiór pomija TIE, NO_PREDICTION i brak odpowiedzi', _s['n_common'] == 2)
check('podział 2×2', (_s['both'], _s['only_personal'], _s['only_baseline'], _s['none']) == (1, 1, 0, 0))

In [ ]:
check('cases.csv nie zmienił się od zamrożenia', file_sha256('cases.csv') == FROZEN_HASH)
validation, session = None, None
if TRYB == 'przyklad':
    ANSWERS = dict(zip([f'V{i:02d}' for i in range(1, 25)], 'A A B A A B A B B A B B B B A B A B A B A A B A'.split()))  # syntetyczne, jak w rozdz. 17
    v = cases.sort_values('display_order')
    validation = pd.DataFrame(dict(participant_id=PARTICIPANT, session_id=SESSION_ID, case_id=v.case_id, pair_id=v.pair_id, display_order=v.display_order,
                                   A_position=v.A_position, answer=v.pair_id.map(ANSWERS), answered_at=now_iso(),
                                   model_version=v.model_version, params=v.params, pred_personal=v.pred_personal,
                                   pred_baseline=v.pred_baseline, pred_rank_sum=v.pred_rank_sum, app_version='synthetic'))
    validation['chosen_position'] = np.where((validation.answer == 'A') == (validation.A_position == 'left'), 'left', 'right')
    session = dict(rating_fit=6, rating_usefulness=5, n_choices=len(validation))
elif os.path.exists('validation.csv'):
    validation = pd.read_csv('validation.csv', dtype={'session_id': str, 'case_id': str})
    validation = validation[validation.session_id == SESSION_ID]
    validation['pair_id'] = validation.case_id
    if os.path.exists(f'session_{SESSION_ID}.json'):
        session = json.load(open(f'session_{SESSION_ID}.json', encoding='utf-8'))
    check('log zawiera wyłącznie zamrożone pary', set(validation.pair_id) <= set(cases.pair_id))
    check('predykcje w logu zgodne z zamrożonymi', (validation.merge(cases, on='pair_id', suffixes=('', '_f'))
                                                   .eval('pred_personal == pred_personal_f and pred_baseline == pred_baseline_f').all()))
    print(f'Wczytano {len(validation)} wyborów z sesji {SESSION_ID}.', '' if session else '⚠ Brak oceny serii (session_*.json).')
else:
    print('Brak validation.csv: najpierw przeprowadź sesję (sekcja 9).')

if validation is not None:
    s = summarize(validation)
    report(s, 'ROC z rankingu', 'równe wagi')
    print('Ocena dopasowania serii (1–7):', session.get('rating_fit') if session else '—',
          '| ocena użyteczności serii (1–7):', session.get('rating_usefulness') if session else '—')
    check_example('ROC 9/12, równe wagi 7/12, 2×2 = 5/4/2/1',
                  (s['hits_personal'], s['hits_baseline'], s['n_common'], s['both'], s['only_personal'], s['only_baseline'], s['none'])
                  == (9, 7, 12, 5, 4, 2, 1))

In [ ]:
# Analiza błędów: pary, w których model indywidualny się pomylił
if validation is not None:
    err = validation[(validation.answer != validation.pred_personal) & ~validation.pred_personal.isin(['TIE', 'NO_PREDICTION'])
                     & validation.answer.notna()]
    for pid in err.pair_id:
        a, b = offers(pool[pool.pair_id == pid].iloc[0])
        row = validation.set_index('pair_id').loc[pid]
        print(f'{pid}: wybór {row.answer}, ROC {row.pred_personal}, suma rang {row.pred_rank_sum}')
        plot_contrib(a, b, w_roc, title=f'{pid}: wkład kryteriów (dopasowanie do profilu, nie jakość hotelu)')

## 11 · Ile informacji niesie test? Pary różnicujące przy wszystkich 120 rankingach
Przewodnik (rozdz. 18) podaje, że przy budżecie 400 zł pula V01–V22 daje od 3 do 10 par różnicujących, z medianą 6, a co najmniej 6 par uzyskano dla 92/120 rankingów. W trybie `wlasne` liczymy to dla budżetu Waszego uczestnika i zaznaczamy jego ranking.

In [ ]:
from itertools import permutations
counts, mine = [], None
for perm in permutations(range(1, 6)):
    rk = dict(zip(CRIT, perm))
    n = None  # TODO: liczba par różnicujących dla rankingu rk (wagi ROC, ten sam budżet)
    counts.append(n)
    if rk == ranks:
        mine = n
counts = np.array(counts)
print(f'min {counts.min()}, max {counts.max()}, mediana {np.median(counts):g}, ≥6: {(counts >= 6).sum()}/120 | ranking uczestnika: {mine}')
check_example('3–10, mediana 6, 92/120 rankingów z ≥ 6 parami', (counts.min(), counts.max(), np.median(counts)) == (3, 10, 6)
              and (counts >= 6).sum() == 92)
plt.hist(counts, bins=np.arange(counts.min() - .5, counts.max() + 1.5)); plt.axvline(mine, color='k', ls='--', label='uczestnik')
plt.xlabel('pary różnicujące'); plt.ylabel('liczba rankingów'); plt.legend(); plt.show()

## 12 · Pytania do raportu
1. Dlaczego do reguły trafiają wagi z rankingu swingów, a nie z deklarowanej ważności 1–7? Porównaj obie informacje u Waszego uczestnika.
2. Ile par różnicowało modele? Co to mówi o sile wniosku z porównania trafności?
3. Którą część tabeli 2×2 trzeba omówić i dlaczego?
4. Czy błędy modelu leżą w parach, w których wagi sumy rang zmieniają predykcję? Co z tego wynika?
5. Czy oceny dopasowania i użyteczności (1–7) zgadzają się z trafnością? Dlaczego samoocena nie zastępuje walidacji?